# ⚾ Train Custom YOLOv8 Blitzball Detector on Google Colab

This notebook trains a custom YOLOv8 deep learning model to accurately detect fast-moving Blitzballs in flight for the **Blitzball Pitch Tracker Pro** suite.

### 📋 Prerequisites & Workflow:
1. **GPU Acceleration**: Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU** (or A100/V100).
2. **Upload Dataset**: Upload `dataset.zip` generated via `python prepare_dataset.py --zip`.
3. **Train Model**: Run training with `epochs=60`, `imgsz=640`, `batch=16`.
4. **Export Weights**: Download `best.pt` / `blitzball_detector.pt` and place it in your local `models/` directory.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 2. Install Ultralytics and Dependencies

In [ ]:
%pip install -q ultralytics opencv-python matplotlib pyyaml

import torch
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:         {torch.cuda.get_device_name(0)}")

## 3. Upload & Unpack Blitzball Dataset

Run the cell below to upload your `dataset.zip` file directly or extract an existing upload.

In [ ]:
import os
import zipfile
from pathlib import Path

# If dataset.zip is not yet uploaded, prompt user or use existing
if not os.path.exists("dataset.zip") and not os.path.exists("dataset/data.yaml"):
    try:
        from google.colab import files
        print("Please upload your 'dataset.zip' file:")
        uploaded = files.upload()
    except Exception as e:
        print("Running outside Colab or manual upload mode.")

# Extract dataset
if os.path.exists("dataset.zip"):
    print("Extracting dataset.zip...")
    with zipfile.ZipFile("dataset.zip", "r") as zip_ref:
        zip_ref.extractall(".")
    print("Dataset extraction complete!")

# Verify data.yaml
yaml_path = "dataset/data.yaml"
if os.path.exists(yaml_path):
    with open(yaml_path, "r") as f:
        print("\n--- data.yaml Contents ---")
        print(f.read())
        print("---------------------------")
else:
    print(f"[Warning] '{yaml_path}' not found. Check dataset folder structure.")

## 4. Initialize Pretrained YOLO Model

We start with the official pretrained `yolov8n.pt` (nano) backbone for real-time high-FPS inference.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model (or yolo11n.pt)
model = YOLO("yolov8n.pt")
print("Loaded pretrained YOLOv8n backbone successfully.")

## 5. Train YOLOv8 Blitzball Detector

Training parameters:
- **Epochs**: `60`
- **Image Size (`imgsz`)**: `640`
- **Batch Size**: `16`
- **Device**: `0` (GPU acceleration)

In [ ]:
# Launch fine-tuning training job
results = model.train(
    data="dataset/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="blitzball_detector",
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,
    save=True,
    workers=2,
    optimizer="auto",
    verbose=True
)

## 6. Evaluate Model Performance & Visualize Metrics

Inspect validation mAP, precision, recall, and training curve plots.

In [ ]:
from IPython.display import Image, display
import glob

# Run validation
metrics = model.val()
print(f"\nValidation Box mAP50:    {metrics.box.map50:.4f}")
print(f"Validation Box mAP50-95: {metrics.box.map:.4f}")

# Display training results curve
result_plots = glob.glob("runs/detect/**/results.png", recursive=True)
if result_plots:
    print(f"\nTraining Curves ({result_plots[-1]}):")
    display(Image(filename=result_plots[-1]))

# Display Confusion Matrix
cm_plots = glob.glob("runs/detect/**/confusion_matrix.png", recursive=True)
if cm_plots:
    print(f"\nConfusion Matrix ({cm_plots[-1]}):")
    display(Image(filename=cm_plots[-1]))

## 7. Test Inference on Sample Validation Pitch Frames

In [ ]:
import cv2
import matplotlib.pyplot as plt

val_images = glob.glob("dataset/images/val/*.jpg")
if val_images:
    sample_images = val_images[:min(4, len(val_images))]
    predictions = model.predict(source=sample_images, conf=0.25, imgsz=640)

    fig, axes = plt.subplots(1, len(sample_images), figsize=(18, 6))
    if len(sample_images) == 1:
        axes = [axes]

    for idx, r in enumerate(predictions):
        annotated_bgr = r.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(annotated_rgb)
        axes[idx].set_title(f"Detections: {len(r.boxes)}")
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No validation images found to test.")

## 8. Export and Package Trained Weights (`best.pt`)

Export `best.pt` as `blitzball_detector.pt` and download to your local machine for use in **Blitzball Pitch Tracker Pro**.

In [ ]:
import shutil
import glob

# Locate best.pt
best_weights = glob.glob("runs/detect/**/weights/best.pt", recursive=True)
if best_weights:
    best_path = best_weights[-1]
    export_name = "blitzball_detector.pt"
    shutil.copy2(best_path, export_name)
    print(f"\n[Success] Exported weights to '{export_name}' (from {best_path})")
    print(f"File size: {os.path.getsize(export_name) / (1024*1024):.2f} MB")

    # Download directly if running on Google Colab
    try:
        from google.colab import files
        print("Initiating download of 'blitzball_detector.pt'...")
        files.download(export_name)
    except Exception:
        print(f"Saved locally as '{export_name}'. Copy to models/blitzball_detector.pt in your project.")
else:
    print("[Error] Could not find best.pt in runs/detect/.")